In [2]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import numpy as np
import pandas as pd
import h5py

import pycbc.conversions, pycbc.distributions, pycbc.waveform, pycbc.filter, pycbc.types, pycbc.psd, pycbc.fft

from tqdm import tqdm
import datetime
import multiprocessing
import uuid
from argparse import ArgumentParser
import logging

class GenUniformWaveform(object):
    '''Waveform Generator
    '''
    def __init__(self, buffer_length, sample_rate, f_lower):
        self.f_lower = f_lower
        self.delta_f = 1.0 / buffer_length
        tlen = int(buffer_length * sample_rate) # buffer length x sample_rate
        self.flen = tlen // 2 + 1

        #psd is hard coded to O3 psd
        psd = pycbc.psd.read.from_txt('/work/yifanwang/ecc/templatebank/o3psd.txt', 
            self.flen, self.delta_f, self.f_lower, is_asd_file = False)
        
        self.kmin = int(f_lower * buffer_length)
        self.w = ((1.0 / psd[self.kmin:-1]) ** 0.5).astype(np.float32)
        
        qtilde = pycbc.types.zeros(tlen, np.complex64) # correlation in Fourier domain
        q = pycbc.types.zeros(tlen, np.complex64) # correlation
        self.qtilde_view = qtilde[self.kmin:self.flen - 1]
        self.ifft = pycbc.fft.IFFT(qtilde, q)
        
        # the maximum is around 0
        self.md = q._data[-100:]
        self.md2 = q._data[0:100] 

    def generate(self, **kwds):
        '''Return normalized hp
        '''
        if kwds['approximant'] in pycbc.waveform.fd_approximants():  
            hp, _ = pycbc.waveform.get_fd_waveform(delta_f = self.delta_f, **kwds)
        else:
            dt = 1.0 / self.sample_rate
            hp = pycbc.waveform.get_waveform_filter(
                        pycbc.types.zeros(self.flen, dtype=np.complex64),
                        delta_f=self.delta_f,
                        delta_t=dt,
                        f_lower=self.f_lower,
                        **kwds)
        
        hp.resize(self.flen)
        hp = hp.astype(np.complex64)
        
        hp[self.kmin:-1] *= self.w
        s = pycbc.filter.sigmasq(hp, low_frequency_cutoff=self.f_lower)
        hp /= s**0.5 
        
        hp.params = kwds
        hp.s = s

        return hp

    def match(self, hp, hc):
        hp.view = hp[self.kmin:-1]
        hc.view = hc[self.kmin:-1]
        pycbc.filter.correlate(hp.view, hc.view, self.qtilde_view)
        self.ifft.execute()
        m = max(abs(self.md).max(), abs(self.md2).max())
        return m * 4.0 * self.delta_f

    def overlap(self, hp, hc):
        o = hp.inner(hc)
        return o * 4.0 * self.delta_f

def wf_wrapper(p):
    index = p['index']
    try:
        hp = gen.generate(**p)
        return index, hp
    except Exception as e:
        return index, None

def match_wrapper(p):
    '''A wrapper function to compute match
    '''
    h1 =pycbc.types.FrequencySeries(initial_array=p['h1_data'], delta_f=p['h1_delta_f'],epoch=p['h1_epoch'])
    h2 =pycbc.types.FrequencySeries(initial_array=p['h2_data'], delta_f=p['h2_delta_f'],epoch=p['h2_epoch'])
    return p['bank_index'], gen.match(h1, h2)

In [4]:
gen = GenUniformWaveform(buffer_length = 32, sample_rate = 2048, f_lower = 20)

In [10]:
with h5py.File('/work/yifanwang/ecc/bank/rollback/rerun-32buffer/buffer32eccbank.hdf') as f:
        df_bank = pd.DataFrame(
           {'mass1': f['mass1'][:],
            'mass2': f['mass2'][:],
            'tau0': pycbc.conversions.tau0_from_mass1_mass2(f['mass1'][:],f['mass2'][:],15),
            'eccentricity': f['eccentricity'][:],
            'rel_anomaly': f['rel_anomaly'][:],
            'spin1z': f['spin1z'][:],
            'spin2z': f['spin2z'][:],
            'approximant': f['approximant'][:].astype('str'),
            'f_lower': f['f_lower'][:]}
        )
df_bank['index'] = df_bank.index

In [11]:
parlist = ['index', 'approximant', 'f_lower', 'mass1', 'mass2', 'spin1z', 'spin2z', 'eccentricity', 'rel_anomaly']

In [12]:
idx = 0
hp = {}
_, hp[idx] = wf_wrapper({k: df_bank.loc[idx,k] for k in parlist})

In [13]:
idx = 1
_, hp[idx] = wf_wrapper({k: df_bank.loc[idx,k] for k in parlist})

In [14]:
hp

{0: <pycbc.types.frequencyseries.FrequencySeries at 0x1523e6b16a50>,
 1: <pycbc.types.frequencyseries.FrequencySeries at 0x1523e692ab90>}

In [17]:
%%timeit
m1 = gen.match(hp[0],hp[1])

25.4 ms ± 469 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [21]:
m1 = gen.match(hp[0],hp[1])

In [22]:
m1

0.17922832071781158

In [31]:
%%timeit
_, m2 = match_wrapper(
    {'bank_index': 0,
                'h1_data': hp[0].data,
                'h1_delta_f': hp[0].delta_f,
                'h1_epoch': hp[0].epoch,
                'h2_data': hp[1].data,
                'h2_delta_f': hp[1].delta_f,
                'h2_epoch': hp[1].epoch
    }
)

25.3 ms ± 121 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [30]:
_, m2 = match_wrapper(
    {'bank_index': 0,
                'h1_data': hp[0].data,
                'h1_delta_f': hp[0].delta_f,
                'h1_epoch': hp[0].epoch,
                'h2_data': hp[1].data,
                'h2_delta_f': hp[1].delta_f,
                'h2_epoch': hp[1].epoch
    }
)

In [32]:
m2

0.17922832071781158